# Portfolio Optimisation & Efficient Frontier

This notebook demonstrates how to:
1. Download historical price data for a basket of stocks.
2. Compute daily returns and annualised statistics.
3. Run a Monte Carlo simulation of random portfolios.
4. Trace the **Efficient Frontier** by minimising volatility for each target return.
5. Identify the **Maximum Sharpe Ratio** portfolio.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure src/ is importable regardless of where the notebook is executed from
sys.path.insert(0, str(Path(os.path.abspath('')).parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    download_price_data,
    compute_daily_returns,
    portfolio_performance,
    simulate_random_portfolios,
    compute_efficient_frontier,
    plot_efficient_frontier,
)

%matplotlib inline

## 1. Configuration

In [ ]:
# ----- User-configurable parameters -----
TICKERS      = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
START_DATE   = "2020-01-01"
END_DATE     = "2024-12-31"
TRADING_DAYS = 252          # annualisation factor
RISK_FREE    = 0.04         # annual risk-free rate (e.g. 4 %)
N_PORTFOLIOS = 5000         # random portfolios for Monte Carlo
N_FRONTIER   = 100          # points on the efficient frontier

## 2. Download Price Data

In [ ]:
prices = download_price_data(TICKERS, START_DATE, END_DATE)
print(f"Downloaded {len(prices)} trading days for {list(prices.columns)}")
prices.tail()

## 3. Compute Returns & Covariance

In [ ]:
daily_returns = compute_daily_returns(prices)
mean_returns  = daily_returns.mean()
cov_matrix    = daily_returns.cov()

print("Annualised mean returns:")
print((mean_returns * TRADING_DAYS).round(4))

## 4. Monte Carlo Simulation

In [ ]:
np.random.seed(42)
simulated = simulate_random_portfolios(
    mean_returns, cov_matrix,
    num_portfolios=N_PORTFOLIOS,
    trading_days=TRADING_DAYS,
    risk_free_rate=RISK_FREE,
)
print(f"Simulated {len(simulated)} random portfolios.")
simulated[["Return", "Volatility", "Sharpe"]].describe().round(4)

## 5. Maximum Sharpe Ratio Portfolio

In [ ]:
max_sharpe_idx = simulated["Sharpe"].idxmax()
best = simulated.iloc[max_sharpe_idx]

print("=== Maximum Sharpe Ratio Portfolio ===")
print(f"  Return     : {best['Return']:.2%}")
print(f"  Volatility : {best['Volatility']:.2%}")
print(f"  Sharpe     : {best['Sharpe']:.4f}")
print("\nWeights:")
for ticker in TICKERS:
    print(f"  {ticker:6s}: {best[ticker]:.2%}")

## 6. Efficient Frontier

In [ ]:
frontier = compute_efficient_frontier(
    mean_returns, cov_matrix,
    num_points=N_FRONTIER,
    trading_days=TRADING_DAYS,
)
print(f"Computed {len(frontier)} frontier points.")

## 7. Visualisation

In [ ]:
fig = plot_efficient_frontier(simulated, frontier, max_sharpe_idx=max_sharpe_idx)
plt.show()

# Persist the figure inside the data directory
output_path = str(Path(os.path.abspath('')).parent / 'data' / 'efficient_frontier.png')
fig.savefig(output_path, dpi=150)
print(f"Figure saved to {output_path}")

## 8. Summary Table

In [ ]:
# Equal-weight benchmark
n = len(TICKERS)
ew_weights = np.ones(n) / n
ew_ret, ew_vol, ew_sharpe = portfolio_performance(
    ew_weights, mean_returns, cov_matrix, TRADING_DAYS
)
ew_sharpe_adj = (ew_ret - RISK_FREE) / ew_vol

summary = pd.DataFrame({
    "Portfolio"  : ["Equal Weight", "Max Sharpe (MC)"],
    "Return"     : [f"{ew_ret:.2%}", f"{best['Return']:.2%}"],
    "Volatility" : [f"{ew_vol:.2%}", f"{best['Volatility']:.2%}"],
    "Sharpe"     : [f"{ew_sharpe_adj:.4f}", f"{best['Sharpe']:.4f}"],
})
summary